# AquaSense AI — Sprint 4 Colab : amélioration MLP (tuning)

**Équipe :** EHTP MIG S4

Tuning complet : MLP + ResidualMLP + 1D-CNN + grid search.

**Prérequis :** `AquaSense_S4_Colab.zip` sur Drive · Runtime **GPU T4** · ~45–90 min

**Baseline précédent :** F1-Macro 0.53 · Recall needs repair 0.60

## 1. Setup (première fois)

Exécute les cellules **1.1** puis **1.2**, puis **Runtime → Restart session**.

> **Ne pas** faire `%pip install numpy` ou `pandas` seuls sur Colab — ça casse TensorFlow (`_blas_supports_fpe`).
> On installe seulement `imbalanced-learn`.

In [ ]:
# 1.1 — Monter Drive
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
# 1.2 — Dézip + pip MINIMAL (pas de upgrade numpy/pandas !)
ZIP = "/content/drive/MyDrive/AquaSense_S4_Colab.zip"  # ajuste si besoin

!unzip -o -q "{ZIP}" -d /content/
%cd /content/AquaSense_S4_Colab

%pip install -q imbalanced-learn

!ls -la data/cleaned/train_clean.csv src/train_dl.py

### STOP — Restart session

Menu : **Runtime → Restart session** (ou Ctrl+M puis `.`)

Ensuite passe à la section **2** (ne ré-exécute pas le `%pip`).

## 2. Après restart — vérification GPU

Exécute **2.1** puis **2.2**.

In [ ]:
# 2.1 — Remonter Drive + aller dans le projet (SANS pip)
from google.colab import drive
drive.mount("/content/drive")

%cd /content/AquaSense_S4_Colab

In [ ]:
# 2.2 — Vérifier TensorFlow + GPU
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import scipy
import tensorflow as tf

PROJECT_ROOT = Path("/content/AquaSense_S4_Colab")
sys.path.insert(0, str(PROJECT_ROOT))

print("numpy  :", np.__version__)
print("scipy  :", scipy.__version__)
print("pandas :", pd.__version__)
print("tf     :", tf.__version__)
print("GPU    :", tf.config.list_physical_devices("GPU"))
print("train_dl :", (PROJECT_ROOT / "src/train_dl.py").exists())

if not tf.config.list_physical_devices("GPU"):
    raise RuntimeError("Pas de GPU — Runtime -> Change runtime type -> T4 GPU")

### Si erreur `_blas_supports_fpe` (numpy/scipy cassés)

Exécute la cellule **2.3 FIX** ci-dessous, puis **Restart session**, puis refais **2.1** et **2.2** seulement.

In [ ]:
# 2.3 FIX — seulement si import tensorflow echoue (numpy/scipy desynchronises)
%pip install -q --upgrade scipy numpy
print("OK — maintenant Runtime -> Restart session, puis section 2.1 et 2.2")

## 3. Tuning complet (~45–90 min)

`python -m src.train_dl tune` → MLP + Residual + CNN + grid search

In [ ]:
%cd /content/AquaSense_S4_Colab
!python -m src.train_dl tune

## 4. Résultats

In [ ]:
import json
from pathlib import Path
import pandas as pd

ROOT = Path("/content/AquaSense_S4_Colab")
csv_path = ROOT / "reports/sprint_04_dl_comparison.csv"
metrics_path = ROOT / "reports/sprint_04_metrics.json"

if csv_path.exists():
    display(pd.read_csv(csv_path).sort_values("f1_macro", ascending=False))

if metrics_path.exists():
    results = json.loads(metrics_path.read_text(encoding="utf-8"))
    champ = results.get("champion_dl", "?")
    m = results["models"].get(champ, {})
    f1, recall, acc = m.get("f1_macro", 0), m.get("recall_needs_repair", 0), m.get("accuracy", 0)
    print(f"Champion DL : {champ}")
    print(f"  F1-Macro            : {f1:.4f}  ({'OK' if f1 >= 0.72 else 'NON'} vs 0.72)")
    print(f"  Recall needs repair : {recall:.4f}  ({'OK' if recall >= 0.65 else 'NON'} vs 0.65)")
    print(f"  Accuracy            : {acc:.4f}")
    print("\nRef ML S3 : RF F1=0.6658 | XGB recall=0.6952 | MLP baseline=0.5297")

## 5. Copier sur Drive

In [ ]:
import shutil
from pathlib import Path

ROOT = Path("/content/AquaSense_S4_Colab")
DRIVE = Path("/content/drive/MyDrive/AquaSense_DL_results")
DRIVE.mkdir(parents=True, exist_ok=True)

files = [
    "mlp_tuned_best_v1.keras", "best_dl_model.keras",
    "residual_mlp_v1.keras", "cnn1d_v1.keras", "mlp_best_v1.keras",
    "sprint_04_metrics.json", "sprint_04_dl_comparison.csv",
]
for name in files:
    src = (ROOT / "models" / name) if name.endswith(".keras") else (ROOT / "reports" / name)
    if src.exists():
        shutil.copy(src, DRIVE / name)
        print("OK ->", DRIVE / name)
    else:
        print("(absent)", name)
print("\nDossier:", DRIVE)